# Vector Retriever

You will create a vector retriever using the Neo4j GraphRAG Python package with AWS Bedrock.

You will be able to review how the vector index is used to retrieve similar results and how the context can be used by an LLM to provide a response.

---

## Setup

Run from CLI first (if not done already):
```bash
./setup-inference-profile.sh haiku
```

Then copy the output ARN and paste it into `INFERENCE_PROFILE_ARN` in the configuration cell below.

## 1. Configuration

Configure your AWS Bedrock inference profile and Neo4j connection details.

In [ ]:
#################################################
# CONFIGURATION
#################################################

# AWS Bedrock Configuration
INFERENCE_PROFILE_ARN = "PASTE_YOUR_ARN_HERE"  # <-- PASTE HERE
REGION = "us-west-2"

# OpenAI Configuration (for embeddings)
# The vector index was created with OpenAI text-embedding-ada-002
OPENAI_API_KEY = "PASTE_YOUR_OPENAI_API_KEY_HERE"  # <-- PASTE HERE

# Neo4j Configuration
NEO4J_URI = "neo4j+s://your-instance.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "your-password"

#################################################

# Validate configuration
errors = []
if "PASTE" in INFERENCE_PROFILE_ARN or "YOUR" in INFERENCE_PROFILE_ARN:
    errors.append("Paste your inference profile ARN (run ./setup-inference-profile.sh)")
if "PASTE" in OPENAI_API_KEY or "YOUR" in OPENAI_API_KEY:
    errors.append("Paste your OpenAI API key")
if "your-instance" in NEO4J_URI:
    errors.append("Update NEO4J_URI with your Neo4j Aura connection string")
if "your-password" in NEO4J_PASSWORD:
    errors.append("Update NEO4J_PASSWORD with your Neo4j password")

if errors:
    print("ERROR: Configuration incomplete!")
    for e in errors:
        print(f"  - {e}")
else:
    print(f"Inference Profile: {INFERENCE_PROFILE_ARN[:60]}...")
    print(f"Region: {REGION}")
    print(f"Neo4j URI: {NEO4J_URI}")
    print("\nConfiguration OK!")

: 

## 2. Install and Verify Packages

In [ ]:
import importlib.metadata

packages = [
    "neo4j",
    "neo4j-graphrag",
    "boto3",
    "openai",
]

print("Required packages:")
print("-" * 50)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:30} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:30} NOT INSTALLED")

In [ ]:
# Install missing packages
%pip install neo4j-graphrag boto3 openai -q

## 3. Imports

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.llm import BedrockLLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings

print("All imports successful!")

## 4. Connect to Neo4j

Create and verify the connection to your Neo4j graph database.

In [ ]:
driver = GraphDatabase.driver(
    NEO4J_URI, 
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)
driver.verify_connectivity()
print("Connected to Neo4j successfully!")

## 5. Initialize LLM and Embedder

Set up the Large Language Model (LLM) and the embedding model.

- **LLM**: Uses AWS Bedrock Claude via the inference profile
- **Embedder**: Uses OpenAI text-embedding-ada-002 (matching the vector index)

In [ ]:
# Initialize LLM using AWS Bedrock
llm = BedrockLLM(
    model_id=INFERENCE_PROFILE_ARN,
    region_name=REGION,
)

# Initialize Embedder using OpenAI (matching the vector index)
embedder = OpenAIEmbeddings(
    api_key=OPENAI_API_KEY,
    model="text-embedding-ada-002",
)

print("LLM and Embedder initialized successfully!")
print(f"LLM Model: {INFERENCE_PROFILE_ARN.split('/')[-1][:50]}...")
print(f"Embedding Model: text-embedding-ada-002 (OpenAI)")

## 6. Initialize Vector Retriever

Set up the vector-based retriever for semantic search over your Neo4j knowledge graph.

> Vector search enables semantic retrieval of text chunks from your Neo4j graph.  
> Instead of keyword matching, it finds the most contextually similar passages to your query, even if the wording is different.

In [ ]:
# Initialize Vector Retriever
vector_retriever = VectorRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    return_properties=['text']
)

print("Vector Retriever initialized!")

The **VectorRetriever** class:
- Connects to the Neo4j database using the provided `driver`.
- Uses the `chunkEmbeddings` vector index for efficient semantic retrieval.
- The `embedder` generates embeddings for the query.
- Returns the `text` property from matching chunks.

> **Tip:**  
> You can modify the `return_properties` list to include additional properties from the retrieved nodes.

---

## 7. Simple Vector Search Diagnostic 

You can use the vector retriever to search for semantically similar data.

Test the vector search by retrieving the top 5 most relevant text chunks from the Neo4j knowledge graph for the given query.

In [ ]:
# Simple Vector Search
query = "What are the risks that Apple faces?"
result = vector_retriever.search(query_text=query, top_k=5)

print(f"Number of results returned: {len(result.items)}\n")
for item in result.items:
    print(f"Score: {item.metadata['score']:.4f}, Content: {item.content[0:100]}..., id: {item.metadata['id']}")

**How it works:**  
1. The example `query`, "What are the risks that Apple faces?", is created
2. `vector_retriever.search()` runs the query and returns the top 5 matches based on vector similarity.
3. The results are formatted displaying:
    * The similarity score (`Score`)
    * A snippet of the retrieved content (`Content`)
    * The unique identifier for each chunk (`id`)

This diagnostic helps you verify that the vector search is working and inspect the quality of the top results for your query.

> **Tip:**
> Inspecting the returned results to verify relevance can help you to adjust your chunking or embedding strategy.

---

## 8. Graph Retrieval-Augmented Generation (GraphRAG) Query

You can use the `GraphRAG` class to create a graph retrieval-augmented generation (GraphRAG) pipeline.

The `GraphRAG` class combines a Large Language Model (LLM) with a vector-based retriever to answer questions using both semantic search and generative reasoning.

In [ ]:
# Initialize GraphRAG and Perform Search
query = "What are the risks that Apple faces?"
rag = GraphRAG(
    llm=llm,
    retriever=vector_retriever
)
response = rag.search(query, retriever_config={"top_k": 5}, return_context=True)

print(f"Number of results returned: {len(response.retriever_result.items)}\n")
print(response.answer)

- **How it works:**  
  1. The retriever (`vector_retriever`) finds the most relevant text chunks from the Neo4j graph based on the input query.
  2. The LLM (`llm`) uses the retrieved context to generate a natural language answer.
  3. The `rag` pipeline is used to `search`.
  4. The `answer` in the `response` is printed

The `GraphRAG` pipeline provides context-aware, accurate answers grounded in your knowledge graph data.

---

## 9. Experiment

Experiment with the vector retriever by modifying the `query`, for example:
- What products does Microsoft reference?
- What warnings have Nvidia given?
- What companies mention AI in their filings?

In [ ]:
# Try a different query
query = "What products does Microsoft reference?"

response = rag.search(query, retriever_config={"top_k": 5}, return_context=True)
print(f"Query: {query}\n")
print(f"Number of results returned: {len(response.retriever_result.items)}\n")
print(response.answer)

---

[Move on to the Advanced Graph RAG: Vector Cypher Retriever Notebook](02_vector_cypher_retriever.ipynb)

In [ ]:
# Cleanup
driver.close()
print("Connection closed.")